#### [You’ve Been Underusing Dataclasses (These Tricks Are Wild)](https://www.youtube.com/watch?v=Y9_h7ehjhO4)

In [1]:
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class Config:
    env: str
    debug: bool=False

a = Config("prod", debug=True)
b = Config("prod")
c = Config("dev", debug=True)

print(a.debug)
print(b.debug)
print(c.debug)


True
False
True


In [ ]:
# Singleton behaviour

from dataclasses import dataclass
from typing import Self, ClassVar

@dataclass(frozen=True, slots=True)
class Config:
    env: str
    debug: bool=False

    _cache: ClassVar[dict[str, Self]] = {}

    @classmethod
    def for_env(cls, env: str, debug: bool=False) -> Self:
        if env not in cls._cache:
            cls._cache[env] = cls(env=env, debug=debug)

        return cls._cache[env]

a = Config.for_env("prod", debug=True)
b = Config.for_env("prod")
c = Config.for_env("dev", debug=True)
d = Config.for_env("test", debug=False)

print(a.debug)
print(b.debug)
print(c.debug)
print(d.debug)

True
True
True
False


In [5]:
# Automatic Registration
from dataclasses import dataclass
from typing import Any, dataclass_transform

REGISTRY: dict[str, type[Any]] = {}

@dataclass_transform()
def event[T](cls:type[T]) -> type[T]:
    REGISTRY[cls.__name__] = dataclass(cls)
    return cls

@event
class UserDeleted:
    user_id: int

@event
class UserCreated:
    user_id: int

e=UserCreated(123)
print(e)
print(REGISTRY)


UserCreated(user_id=123)
{'UserDeleted': <class '__main__.UserDeleted'>, 'UserCreated': <class '__main__.UserCreated'>}
